# Part E · Hardware comparison

Run this notebook from the repository root or `analysis/`. It audits complete official Docker
CORE runs, preserves the exact session and version labels, and exports the report tables
and figures. Five or more distinct replicates of every CORE configuration are required
for a complete session. Where there are multiple complete sessions on one machine for one
revision, the most recent session supplies the timing summary.

## E.1 · Coverage, source identity and operation counters

The exact-version audit groups by the raw recorded commit. The source-cohort audit is shown
separately. It groups revisions only when Git reports identical objects for `indexes/`,
`bench/core_suite.py`, `bench/timing.py`, `loaders/`, `Dockerfile` and `requirements.txt`.
These cover the index, workloads, dataset acquisition, timer and environment definition.
The manifest keeps the runtime Python version and hash seed visible for comparison.
For each index and workload, check counter values before comparing timings. Reference
indexes intentionally leave their counters blank.
If Python versions or hash seeds differ, the timing ratios describe the observed systems and cannot isolate hardware alone.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "analysis/member_c_analysis.py").is_file())
sys.path.insert(0, str(repo))
from analysis import member_c_analysis as c

rows, machines = c.load_data(repo)
out = Path(os.environ.get("DS4300_ANALYSIS_DIR", repo / "analysis"))
figures = out / "figures_member_c"
tables = out / "tables_member_c"
for folder in (figures, tables):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.grid": True, "grid.alpha": 0.2})

core, manifest, exact, cohorts, exact_timings, cohort_timings = c.core_summaries(rows, repo)
display(machines)
display(manifest)
coverage = cohorts.pivot(index="machine_id", columns="source_cohort", values="code_version")
phases = ["Original source", "Current source"]
phases += [label for label in coverage.columns if label not in phases]
coverage = coverage.reindex(index=machines["machine_id"], columns=phases)
display(coverage.fillna("Pending"))
coverage.to_csv(tables / "CORE_coverage.csv")
manifest.to_csv(tables / "CORE_sessions.csv", index=False)
exact_timings.to_csv(tables / "CORE_timings_exact_versions.csv", index=False)
cohort_timings.to_csv(tables / "CORE_timings_source_cohorts.csv", index=False)

for label, sessions in [("code_version", exact), ("source_cohort", cohorts)]:
    audit = c.counter_audit(core, sessions, label)
    audit.to_csv(tables / f"CORE_counters_{label}.csv", index=False)
    coverage = audit.groupby([label, "machines", "distinct_values"]).size().rename("counter_checks")
    display(coverage.reset_index())
    display(audit[audit["distinct_values"].gt(1)])

sources = c.source_objects(repo, manifest["code_version"], c.CORE_SOURCE)
sources.to_csv(tables / "CORE_source_objects.csv", index=False)
display(sources.assign(git_object=sources["git_object"].str[:12]).pivot(
    index="path", columns="code_version", values="git_object"
))


## E.2 · Timing, fastest/slowest ratio, rank and variation

Convert each batch to ns per operation before computing per-machine summaries. Compare medians
for the same workload and configuration, keeping both build sizes separate. The fastest/slowest
ratio is the largest median divided by the smallest. A cohort with one measured machine has no
cross-machine ratio. Rank 1 is fastest among the seven measured implementations, including the
two C-backed references. The per-configuration CV is 100 × sample standard deviation / mean
across timing replicates. The CV table summarizes the 42 configurations per machine.

In [ ]:
exact_ratios = c.core_ratios(exact_timings, "code_version")
ratios = c.core_ratios(cohort_timings)
exact_ratios.to_csv(tables / "CORE_ratios_exact_versions.csv", index=False)
ratios.to_csv(tables / "CORE_ratios_source_cohorts.csv", index=False)

cv = cohort_timings.groupby(["source_cohort", "machine_id"])["cv_percent"].agg(
    configurations="size", median_cv_pct="median",
    q3_cv_pct=lambda x: x.quantile(0.75), max_cv_pct="max"
)
cv.to_csv(tables / "CORE_cv_by_machine.csv")
display(cv.round(2))

if ratios.empty:
    print("Cross-machine timings are pending for every source cohort.")
else:
    display(ratios.groupby("source_cohort")["slowest_over_fastest"].agg(["min", "median", "max"]))
    display(ratios.sort_values("slowest_over_fastest", ascending=False).head(10))

paired = c.core_change(core, cohort_timings)
paired.to_csv(tables / "CORE_before_after.csv", index=False)
display(paired[paired["index_type"].eq("bplus_tree")
               & paired["operation"].isin(["range", "prefix"])].round(3))


In [ ]:
if not ratios.empty:
    ratios = ratios.assign(workload=c.workload_label(ratios))
    rank = cohort_timings.assign(workload=c.workload_label(cohort_timings))
    workload_order = ["Build · 10,000 rows", "Build · 114,000 rows", "Point hit",
                      "Point miss", "Tempo range", "Title prefix"]

    def heatmap(ax, table, palette, low, high, fmt, threshold):
        ax.imshow(table, cmap=palette, vmin=low, vmax=high, aspect="auto")
        ax.set_xticks(range(len(table.columns)), table.columns, rotation=28, ha="right")
        ax.set_yticks(range(len(table.index)), table.index)
        ax.grid(False)
        for row, values in enumerate(table.to_numpy()):
            for col, value in enumerate(values):
                if pd.notna(value):
                    color = "white" if value >= threshold else "#243341"
                    ax.text(col, row, format(value, fmt), ha="center", va="center",
                            color=color, fontsize=9)

    for cohort, part in ratios.groupby("source_cohort"):
        table = part.pivot(index="index_type", columns="workload",
                           values="slowest_over_fastest").reindex(index=c.INDEXES, columns=workload_order)
        high = max(1.5, table.max().max())
        fig, ax = plt.subplots(figsize=(8.4, 3.8), layout="constrained")
        heatmap(ax, table, "Blues", 1, high, ".2f", 1 + (high - 1) * 0.63)
        ax.set_title(f"{cohort} | Slowest / fastest median time (×)")
        slug = cohort.lower().replace(" ", "_").replace("(", "").replace(")", "")
        fig.savefig(figures / f"CORE_speed_ratio_{slug}.png", dpi=180, bbox_inches="tight")
        plt.show()

        ranks = rank[rank["source_cohort"].eq(cohort)]
        rank_tables = list(ranks.groupby("machine_id"))
        fig, axes = plt.subplots(len(rank_tables), 1, figsize=(8.4, 2.9 * len(rank_tables)),
                                 layout="constrained", squeeze=False)
        for ax, (machine, block) in zip(axes.flat, rank_tables):
            table = block.pivot(index="index_type", columns="workload",
                                values="rank").reindex(index=c.INDEXES, columns=workload_order)
            heatmap(ax, table, "PuBu", 1, 7, ".0f", 5)
            ax.set_title(machine, loc="left")
        fig.suptitle(f"{cohort} | Index ranking by median time (1 is fastest)",
                     fontsize=13, weight="bold")
        fig.savefig(figures / f"CORE_ranks_{slug}.png", dpi=180, bbox_inches="tight")
        plt.show()


## E.3 · Index allocations and RAM

E1 measures incremental retained Python allocations of the index once per configuration.
The timing replicates repeat that memory value. It does not measure the full process,
Spotify objects already loaded, peak allocation during a resize or Docker's other memory
use. The projected row count at each Docker budget scales the 114,000-row measurement
linearly. Treat that as a rough upper bound for an index-only footprint. Hash-table
resizes, key duplication, and the host workload can change the actual limit.

In [ ]:
E1_SESSION = None

memory_manifest, memory_selected, memory, budget = c.e1_memory(rows, machines, E1_SESSION)
memory.to_csv(tables / "E1_index_memory.csv", index=False)
budget.to_csv(tables / "E1_docker_ram_estimates.csv", index=False)
display(memory_manifest)
display(memory_selected)
display(budget[["index_type", "comparison_machine", "memory_mib", "docker_memory_gb",
                "docker_budget_pct", "rows_at_full_budget"]].round(3))

if not memory.empty:
    fig, ax = plt.subplots(figsize=(8.4, 4.6), layout="constrained")
    for index, curve in memory.groupby("index_type"):
        curve = curve.sort_values("n_keys")
        ax.plot(curve["n_keys"], curve["memory_mib"], marker="o", markersize=4, label=index)
    ax.set(xscale="log", yscale="log", xlabel="Track ID rows", ylabel="Incremental index memory (MiB)",
           title="E1 | Incremental memory allocated by each index")
    ax.legend(frameon=False, ncol=2)
    fig.savefig(figures / "E1_hardware_memory.png", dpi=180, bbox_inches="tight")
    plt.show()
